## Step 1: Import Libraries and Load Data

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
import joblib
import pickle

# Load the dataset
df = pd.read_csv('adult 3.csv')

## Step 2: Clean Data and Feature Selection
This is the most critical step to fix all errors.
1.  Clean the column names to remove hidden spaces.
2.  Handle missing values.
3.  Remove all columns that cause data leakage.

In [4]:
# --- CRITICAL FIX 1: Clean column names --- 
df.columns = df.columns.str.strip()
print("✅ Column names cleaned.")

# Replace '?' with NaN and drop rows with missing values
df.replace('?', np.nan, inplace=True)
df.dropna(inplace=True)
print("✅ Missing values handled.")

# --- CRITICAL FIX 2: Define features by dropping ALL leaking columns ---
columns_to_drop = ['income', 'relationship']
if 'education' in df.columns:
    columns_to_drop.append('education')
if 'income_encoded' in df.columns:
    columns_to_drop.append('income_encoded')

X = df.drop(columns=columns_to_drop)
y = df['income']

print("✅ Correct features selected for training:")
print(X.columns)

✅ Column names cleaned.
✅ Missing values handled.
✅ Correct features selected for training:
Index(['age', 'workclass', 'fnlwgt', 'educational-num', 'marital-status',
       'occupation', 'race', 'gender', 'capital-gain', 'capital-loss',
       'hours-per-week', 'native-country'],
      dtype='object')


## Step 3: Train-Test Split
Always split the data before any preprocessing.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Data split into {X_train.shape[0]} training samples and {X_test.shape[0]} testing samples.")

Data split into 36177 training samples and 9045 testing samples.


## Step 4: Preprocessing (Label Encoding & Scaling)
This must be done AFTER the split.

In [6]:
# Identify column types from the training set
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(include=np.number).columns

# --- Label Encoding --- 
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    le_dict[col] = le
print("✅ Categorical features encoded.")

# Encode target variable
le_y = LabelEncoder()
y_train = le_y.fit_transform(y_train)
y_test = le_y.transform(y_test)

# --- Scaling --- 
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])
print("✅ Numerical features scaled.")

✅ Categorical features encoded.
✅ Numerical features scaled.


## Step 5: Handle Class Imbalance
Apply SMOTE only on the training data.

In [7]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("✅ Class imbalance handled with SMOTE.")

✅ Class imbalance handled with SMOTE.


/Users/manvidhamija/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


## Step 6: Train Final Model

In [8]:
gbc = GradientBoostingClassifier(random_state=42)
gbc.fit(X_train_res, y_train_res)
print("✅ Final model trained successfully.")

✅ Final model trained successfully.


## Step 7: Save All Artifacts
Save the trained model, the scaler, and the dictionary of label encoders.

In [9]:
# Save the Model
joblib.dump(gbc, "best_model.pkl")
print("-> Model saved to 'best_model.pkl'")

# Save the Scaler
joblib.dump(scaler, "scaler.pkl")
print("-> Scaler saved to 'scaler.pkl'")

# Save the Label Encoders
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(le_dict, f)
print("-> Label encoders saved to 'label_encoders.pkl'")

-> Model saved to 'best_model.pkl'
-> Scaler saved to 'scaler.pkl'
-> Label encoders saved to 'label_encoders.pkl'
